# Week 1 — LLM Fundamentals & Environment Setup

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-01-llm-fundamentals-content.html`. Run the cells in order during lab time.

**You will practice:**
1. Making your first Gemini API call from Python.
2. Seeing `temperature`, `top_p`, and `top_k` change model output on the *same* prompt.
3. Counting tokens for real and comparing it to the "1 token ≈ 4 characters" rule of thumb.
4. Making the *same* call through three tools: the raw SDK, a minimal Google ADK agent, and a minimal LangChain chat model.
5. Two open exercises.

**Before you start:** create a free API key at [Google AI Studio](https://aistudio.google.com/app/apikey) and save it in a `.env` file next to this notebook:

```
GOOGLE_API_KEY="paste-your-key-here"
```


In [13]:
pip install langchain-ollama

     ---------------------------------------- 0.0/43.1 kB ? eta -:--:--
     ---------------------------------------- 43.1/43.1 kB 1.0 MB/s eta 0:00:00
     ---------------------------------------- 0.0/46.7 kB ? eta -:--:--
     ---------------------------------------- 46.7/46.7 kB ? eta 0:00:00
   ---------------------------------------- 0.0/570.0 kB ? eta -:--:--
   ------- -------------------------------- 112.6/570.0 kB 3.2 MB/s eta 0:00:01
   ------------------------------- -------- 450.6/570.0 kB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 570.0/570.0 kB 5.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/744.6 kB ? eta -:--:--
   ----------------------------- --------- 563.2/744.6 kB 34.6 MB/s eta 0:00:01
   --------------------------------------- 744.6/744.6 kB 11.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/158.8 kB ? eta -:--:--
   ---------------------------------------- 158.8/158.8 kB 4.8 MB/s eta 0:00:00
   -------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\yosti\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Environment check

Load the API key and make one call to confirm everything is wired up.

In [6]:
import ollama

MODEL = "qwen2.5:14b"

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "In one sentence, what is a large language model?",
        }
    ],
)

print(response.message.content)

A large language model is an artificial intelligence system designed to process and generate human-like text based on a vast amount of learned data from the internet and other sources.


## 2. Sampling parameters: temperature, top-p, top-k

We'll send the **same prompt** several times, changing only `temperature`. Watch how the output goes from
predictable to varied.

In [7]:
import ollama

MODEL = "qwen2.5:14b"
PROMPT = "Give me one creative name for a coffee shop. Reply with just the name."

for temp in [0.0, 0.4, 0.9, 1.5]:
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        options={
            "temperature": temp,
            "num_predict": 20,  # Equivale a max_output_tokens
        },
    )
    print(f"temperature={temp:<4} -> {response.message.content.strip()}")

temperature=0.0  -> Mocha Muse
temperature=0.4  -> Mug满星
temperature=0.9  -> Bean Voyage
temperature=1.5  -> Mocha Muse


Run the cell above a few times. At `temperature=0.0` the answer should barely change between runs.
At `temperature=1.5` you should see real variety (and occasionally something a little unhinged — that's expected).

Now let's isolate `top_p` and `top_k` by holding temperature fixed at a mid-range value.

In [ ]:
for top_p in [0.1, 0.5, 0.95]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=0.9, top_p=top_p, max_output_tokens=20),
    )
    print(f"top_p={top_p:<5} -> {response.text.strip()}")

print()
for top_k in [1, 5, 40]:
    response = client.models.generate_content(
        model=MODEL,
        contents=PROMPT,
        config=types.GenerateContentConfig(temperature=0.9, top_k=top_k, max_output_tokens=20),
    )
    print(f"top_k={top_k:<3} -> {response.text.strip()}")

## 3. Tokens: count them for real

The rule of thumb is "1 token ≈ 4 characters ≈ 0.75 words" for English. Let's check it against the real
tokenizer using `count_tokens`.

In [12]:
import ollama

MODEL = "qwen2.5:14b"

samples = [
    "Hi!",
    "The capital of France is Paris.",
    "AI Agentic Engineering is a course about building autonomous systems with large language models, "
    "retrieval-augmented generation, and multi-agent orchestration frameworks like Google ADK and LangGraph.",
]

for text in samples:
    res = ollama.generate(
        model=MODEL,
        prompt=text,
        options={"num_predict": 1},
    )

    chars = len(text)
    tokens = res.prompt_eval_count
    ratio = chars / tokens if tokens else 0

    print(
        f"chars={chars:<4} tokens={tokens:<4} chars/token={ratio:.2f}  | {text[:50]!r}"
    )

chars=3    tokens=31   chars/token=0.10  | 'Hi!'
chars=31   tokens=36   chars/token=0.86  | 'The capital of France is Paris.'
chars=200  tokens=65   chars/token=3.08  | 'AI Agentic Engineering is a course about building '


## 4. Same call, three tools

Below, the identical prompt is sent through: **(A)** the raw `google-genai` SDK, **(B)** a one-line **Google ADK**
agent run with `InMemoryRunner`, and **(C)** a **LangChain** chat model. This is a *light preview* — we are not
building real agents yet (tools, multi-step reasoning, ReAct) until Corte 2, Week 6 onward. The goal today is
just to recognize the same underlying API call under each interface.

In [14]:
# --- A. Raw SDK (Ollama Nativo) ---
import ollama

prompt = "Name one famous mathematician and the theorem they're best known for."
MODEL = "qwen2.5:14b"

response = ollama.chat(
    model=MODEL, messages=[{"role": "user", "content": prompt}]
)
print("A) raw SDK:\n", response["message"]["content"])

A) raw SDK:
 One famous mathematician is Pythagoras, and he is best known for the Pythagorean theorem. This theorem states that in a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. Mathematically, it is expressed as \(a^2 + b^2 = c^2\), where \(c\) represents the length of the hypotenuse, and \(a\) and \(b\) represent the lengths of the other two sides.


In [15]:
# --- B. Google ADK (minimal agent) ---
import ollama


class LocalADKAgent:

    def __init__(self, model: str, name: str, instruction: str):
        self.model = model
        self.name = name
        self.instruction = instruction

    def run(self, user_prompt: str) -> str:
        res = ollama.chat(
            model=self.model,
            messages=[
                {"role": "system", "content": self.instruction},
                {"role": "user", "content": user_prompt},
            ],
        )
        return res["message"]["content"]


adk_agent = LocalADKAgent(
    model=MODEL,
    name="week1_demo_agent",
    instruction="Answer concisely, in 1-2 sentences.",
)


def ask_adk_agent(agent, prompt):
    """Small reusable helper: runs one prompt through an ADK agent and returns the final text."""
    return agent.run(prompt)


print(
    "B) Google ADK agent (Local simulation):\n", ask_adk_agent(adk_agent, prompt)
)

B) Google ADK agent (Local simulation):
 Pythagoras is best known for the Pythagorean theorem, which states that in a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides.


In [16]:
# --- C. LangChain chat model ---
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODEL)
lc_response = llm.invoke(prompt)
print("C) LangChain:\n", lc_response.content)

C) LangChain:
 One famous mathematician is Pythagoras, and he is best known for the Pythagorean theorem. This theorem states that in a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. This can be expressed as the equation \(a^2 + b^2 = c^2\), where \(c\) is the length of the hypotenuse, and \(a\) and \(b\) are the lengths of the other two sides.


## 5. Exercises

Complete both before the peer code review activity.

In [17]:
import ollama

MODEL = "qwen2.5:14b"
MY_PROMPT = "Propose a name and a one-sentence tagline for a cybersecurity SaaS platform."

temperatures = [0.0, 0.3, 0.8, 1.4]

print(f"PROMPT: {MY_PROMPT}\n" + "=" * 60)

for temp in temperatures:
    res = ollama.generate(
        model=MODEL,
        prompt=MY_PROMPT,
        options={"temperature": temp, "num_predict": 30},
    )
    print(f"Temp {temp:.1f} | Output: {res['response'].strip()}")

PROMPT: Propose a name and a one-sentence tagline for a cybersecurity SaaS platform.
Temp 0.0 | Output: **Name:** SentinelGuard

**Tagline:** "Protecting your digital fortress with intelligent cybersecurity solutions."
Temp 0.3 | Output: **Name:** CyberGuard Pro

**Tagline:** "Unmatched Protection: Secure Your Digital Future with CyberGuard Pro."
Temp 0.8 | Output: Name: SentinelSec

Tagline: "Guarding the digital frontier with intelligent, adaptive security solutions."
Temp 1.4 | Output: **Name:** FortressEdge

**Tagline:** Secure your digital landscape with FortressEdge's proactive cybersecurity solutions.


In [18]:
import ollama

MODEL = "qwen2.5:14b"
samples = [
    "Hi!",
    "Software quality assurance ensures system reliability and performance standards.",
    "ISO/IEC 25010 defines system and software quality models, evaluating characteristics such as functional suitability, performance efficiency, and security.",
]


def estimate_tokens(text: str) -> float:
    return len(text) / 4.0


for text in samples:
    res = ollama.generate(model=MODEL, prompt=text, options={"num_predict": 1})
    real = res["prompt_eval_count"]
    estimate = estimate_tokens(text)

    error_pct = (
        (abs(estimate - real) / real) * 100 if real > 0 else 0.0
    )

    print(
        f"Text: '{text[:30]}...' | Real: {real} | Est: {estimate:.1f} | Error: {error_pct:.1f}%"
    )

Text: 'Hi!...' | Real: 31 | Est: 0.8 | Error: 97.6%
Text: 'Software quality assurance ens...' | Real: 39 | Est: 20.0 | Error: 48.7%
Text: 'ISO/IEC 25010 defines system a...' | Real: 59 | Est: 38.5 | Error: 34.7%


## Next week

Week 2 — **Context Engineering I**: system prompts, few-shot prompting, chain-of-thought, and forcing structured
JSON output. See `week-02-context-engineering-i-content.html`.